# Feature Engineering
Adds non-model derived features used by downstream analysis notebooks.

In [1]:
import os
import pandas as pd
import numpy as np

INPUT_FILE = os.getenv("INPUT_FILE", "Processed_Reviews.csv")
OUTPUT_FILE = os.getenv("OUTPUT_FILE", "Processed_Reviews.csv")
print(f"Using input: {INPUT_FILE} -> output: {OUTPUT_FILE}")

Using input: Processed_Reviews.csv -> output: Processed_Reviews.csv


In [2]:
# --------------------------------------------
# 1. Load enriched dataset
# --------------------------------------------
df = pd.read_csv(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "Review_Length",
    "Helpful_Votes",
    "User_Contributions",
    "Combined_Sentiment",
    "Location_Name",
    "Rating",
 ]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

Loaded: Processed_Reviews.csv
Shape: (16156, 28)
Columns: ['Location_Name', 'Located_City', 'Province', 'District', 'Location_Type', 'User_Locale', 'User_Country', 'User_Region', 'Travel_Date_Month', 'Travel_Date_Year', 'Published_Date_Month', 'Published_Date_Year', 'User_Contributions', 'Rating', 'Helpful_Votes', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days', 'Combined_Text', 'Combined_Sentiment', 'Sentiment_Score', 'Emotion', 'Dominant_Topic', 'Topic_Probability', 'Topic_Keywords', 'Review_Theme', 'Sentiment_Rating_Gap']


In [3]:
# --------------------------------------------
# 2. Derived features
# --------------------------------------------
# Length bucket
df["Length_Bucket"] = pd.cut(
    df["Review_Length"].fillna(0),
    bins=[-1, 50, 150, 300, np.inf],
    labels=["Very Short", "Short", "Medium", "Long"]
)

# Helpfulness
df["Has_Helpful_Votes"] = (df["Helpful_Votes"].fillna(0) > 0).astype(int)
df["Helpfulness_Ratio"] = df["Helpful_Votes"].fillna(0) / (df["User_Contributions"].fillna(0) + 1)
df["Helpful_Bucket"] = pd.cut(
    df["Helpful_Votes"].fillna(0),
    bins=[-1, 0, 5, 20, np.inf],
    labels=["None", "Low", "Medium", "High"]
)

# Reviewer experience
df["Reviewer_Experience"] = pd.cut(
    df["User_Contributions"].fillna(0),
    bins=[-1, 10, 50, 200, np.inf],
    labels=["New", "Casual", "Active", "Expert"]
)

# Sentiment numeric
sentiment_map = {"NEGATIVE": -1, "NEUTRAL": 0, "POSITIVE": 1}
df["Sentiment_Numeric"] = df["Combined_Sentiment"].map(sentiment_map)

# Destination aggregates
df["Location_Avg_Rating"] = df.groupby("Location_Name")["Rating"].transform("mean")
df["Location_Review_Count"] = df.groupby("Location_Name")["Rating"].transform("count")
df["Location_Sentiment_Mean"] = df.groupby("Location_Name")["Sentiment_Numeric"].transform("mean")

# Ranking
df["Rank_By_Rating"] = df["Location_Avg_Rating"].rank(ascending=False, method="dense")
df["Rank_By_Popularity"] = df["Location_Review_Count"].rank(ascending=False, method="dense")
df["Popularity_Quality_Gap"] = df["Rank_By_Popularity"] - df["Rank_By_Rating"]

# Review quality score
review_len = df["Review_Length"].fillna(0)
helpful_votes = df["Helpful_Votes"].fillna(0)
review_len_norm = (review_len - review_len.min()) / (review_len.max() - review_len.min() + 1e-9)
helpful_votes_norm = (helpful_votes - helpful_votes.min()) / (helpful_votes.max() - helpful_votes.min() + 1e-9)
df["Review_Quality_Score"] = 0.5 * review_len_norm + 0.5 * helpful_votes_norm

In [4]:
# --------------------------------------------
# 3. Save final dataset
# --------------------------------------------
new_cols = [
    "Length_Bucket",
    "Has_Helpful_Votes",
    "Helpfulness_Ratio",
    "Helpful_Bucket",
    "Reviewer_Experience",
    "Sentiment_Numeric",
    "Location_Avg_Rating",
    "Location_Review_Count",
    "Location_Sentiment_Mean",
    "Rank_By_Rating",
    "Rank_By_Popularity",
    "Popularity_Quality_Gap",
    "Review_Quality_Score",
 ]
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Saved: {OUTPUT_FILE}")
print("Final shape:", df.shape)
print("\nNew columns created:")
for col in new_cols:
    print(f"- {col}")
print("\nPreview:")
print(df[new_cols].head())


✅ Saved: Processed_Reviews.csv
Final shape: (16156, 41)

New columns created:
- Length_Bucket
- Has_Helpful_Votes
- Helpfulness_Ratio
- Helpful_Bucket
- Reviewer_Experience
- Sentiment_Numeric
- Location_Avg_Rating
- Location_Review_Count
- Location_Sentiment_Mean
- Rank_By_Rating
- Rank_By_Popularity
- Popularity_Quality_Gap
- Review_Quality_Score

Preview:
  Length_Bucket  Has_Helpful_Votes  Helpfulness_Ratio Helpful_Bucket  \
0          Long                  1           0.111111            Low   
1          Long                  0           0.000000           None   
2        Medium                  0           0.000000           None   
3        Medium                  0           0.000000           None   
4        Medium                  0           0.000000           None   

  Reviewer_Experience  Sentiment_Numeric  Location_Avg_Rating  \
0                 New                  1             4.058824   
1                 New                  1             4.058824   
2         